# University Exam Timetabling — Interactive Execution Walkthrough
**Repository:** `https://github.com/Walid2003H/Exam-timetabling`  
**Scope:** Inspect the modular software architecture (`P1–P8`) and full Git repository tree, validate the 12 isolated benchmark instances, execute the six evolutionary and swarm intelligence algorithms (base and memetic variants) under a unified Function Evaluation Equivalent (`FEE`) budget, and generate non-parametric statistical comparisons and figures.

## 1. Repository Synchronization & Modular Architecture (`P1–P8`)
The codebase is organized into a layered modular architecture that decouples the Compressed Sparse Row (`CSR`) conflict-graph representation, objective and constraint-violation kernels, shared local search operators, and metaheuristic solvers. All solvers share a unified `Solver` interface and a strict `Budget` accounting mechanism.

The cell below clones or synchronizes the repository to the latest branch and summarizes the responsibility of each core module.

In [ ]:
# Step 1: Clone or update repository and inspect modular architecture (P1-P8)
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Walid2003H/Exam-timetabling.git"
REPO_DIR = Path("Exam-timetabling")

if not REPO_DIR.exists():
    print(f"Cloning repository from {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Updating existing repository at '{REPO_DIR}' ...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=False)

# Automatically align to the branch carrying the latest P1-P8 commits (main or develop)
branches_out = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "-r"], text=True
)
if "origin/develop" in branches_out:
    ahead_count = int(
        subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "rev-list", "--count", "HEAD..origin/develop"],
            text=True,
        ).strip()
    )
    if ahead_count > 0:
        print(f"Switching to 'develop' ({ahead_count} newer commit(s) than current HEAD) ...")
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "-B", "develop", "origin/develop"],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

commit_hash = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
branch_name = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()
print(f"Active branch: '{branch_name}' | Commit: {commit_hash}\n")

architecture_map = [
    ("src/core/", "Problem formulation, CSR Instance, Carter cost, repair, Budget (FEE), Tracker"),
    ("src/data/", "Synthetic planted-solution generator, Toronto parser, SHA-256 & structural validator"),
    ("src/baselines/", "Constructive DSATUR, Random Search, Simulated Annealing, Exact Branch & Bound"),
    ("src/local_search/", "Shared budget-guarded Move, Swap, Kempe-chain, and Hybrid local search operators"),
    ("src/algorithms/", "GA, PSO (Random-Keys & Discrete), ACO (MMAS), and their Memetic hybrids"),
    ("configs/", "Default algorithm hyperparameters and isolated scale-specific tuning presets"),
    ("experiments/", "Unified experiment runner, hyperparameter tuning, and ablation configurations"),
    ("analysis/", "Non-parametric statistical tests (Friedman, Wilcoxon-Holm) and figure generators"),
    ("tests/", "Contract, core math, data integrity, baseline, and metaheuristic unit tests"),
]

print(f"{'Directory':<20} | {'Files':<5} | {'Architectural Responsibility'}")
print("-" * 98)
for rel_dir, desc in architecture_map:
    folder_path = REPO_DIR / rel_dir.rstrip("/")
    py_count = (
        len([p for p in folder_path.rglob("*") if p.is_file() and "__pycache__" not in str(p)])
        if folder_path.exists()
        else 0
    )
    print(f"{rel_dir:<20} | {py_count:<5} | {desc}")

## 2. Complete Git Repository Tree & Commit History
To provide complete structural transparency across all project deliverables, the cell below queries Git directly (`git ls-files` and `git log`) to display the active branches, recent commit history, and the full hierarchical file tree of the repository.

In [ ]:
# Step 2: Display complete Git-tracked directory/file tree and recent commit history
import subprocess

print("=== GIT BRANCHES & RECENT COMMIT HISTORY ===")
branches = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "-a"], text=True
).strip()
print(branches)

print("\nRecent Commits (last 10):")
git_log = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "--oneline", "--decorate", "-n", "10"], text=True
).strip()
print(git_log)

# Query all files tracked by Git (clean repository structure without untracked artifacts)
tracked_files = sorted(
    subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "ls-files"], text=True
    ).splitlines()
)

# Build nested dictionary representing the full directory tree
tree_root = {}
for rel_path in tracked_files:
    parts = rel_path.split("/")
    node = tree_root
    for part in parts[:-1]:
        node = node.setdefault(part, {})
    node[parts[-1]] = None  # Leaf file


def _count_files(node):
    total = 0
    for val in node.values():
        total += 1 if val is None else _count_files(val)
    return total


def _render_tree(node, prefix=""):
    # Sort directories first, then files alphabetically
    entries = sorted(node.items(), key=lambda kv: (kv[1] is None, kv[0].lower()))
    for idx, (name, child) in enumerate(entries):
        is_last = idx == len(entries) - 1
        connector = "└── " if is_last else "├── "
        if child is None:
            print(f"{prefix}{connector}{name}")
        else:
            n_sub = _count_files(child)
            print(f"{prefix}{connector}{name}/ ({n_sub} files)")
            extension = "    " if is_last else "│   "
            _render_tree(child, prefix + extension)


print(f"\n=== COMPLETE GIT REPOSITORY STRUCTURE ({REPO_DIR.name}/) ===")
print(f"{REPO_DIR.name}/ ({len(tracked_files)} tracked files)")
_render_tree(tree_root)

py_files = [p for p in tracked_files if p.endswith(".py")]
print("-" * 72)
print(f"Total Git-tracked files : {len(tracked_files)}")
print(f"Total Python (.py) files: {len(py_files)}")

## 3. Environment Verification & `Numba` JIT Warm-Up
To avoid redundant package reinstallations in `Google Colab`, this step verifies required dependencies, imports the core evaluation and solver modules, and performs an initial `Numba JIT` compilation warm-up pass so first-call compilation overhead does not affect solver runtime measurements.

In [ ]:
# Step 3: Fast dependency check, module imports, and Numba JIT warm-up
import importlib
import importlib.util
import inspect
import pkgutil
import subprocess
import sys
import time

t_env = time.perf_counter()

required_modules = {
    "numpy": "numpy",
    "scipy": "scipy",
    "numba": "numba",
    "matplotlib": "matplotlib",
    "yaml": "pyyaml",
}
missing = [pip_pkg for mod, pip_pkg in required_modules.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

repo_abs = str(REPO_DIR.resolve())
if repo_abs not in sys.path:
    sys.path.insert(0, repo_abs)

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

import src.data as data_pkg
from src.core.solver import Budget
from src.core.problem import proximity_cost, violation_weight
from experiments.run import get_solver

# Dynamically resolve loader, validator, and stats helpers across src.data submodules
_data_modules = [data_pkg]
for _, mod_name, _ in pkgutil.iter_modules(data_pkg.__path__):
    _data_modules.append(importlib.import_module(f"src.data.{mod_name}"))


def _find_callable(candidates):
    for mod in _data_modules:
        for fn_name in candidates:
            fn = getattr(mod, fn_name, None)
            if callable(fn):
                return fn
    return None


_raw_loader = _find_callable(["load_instance_npz", "load_npz", "load_instance", "read_instance"])
_raw_validator = _find_callable(["validate_instance", "check_instance"])
compute_instance_stats = _find_callable(["compute_instance_stats", "instance_stats"])

assert _raw_loader is not None, f"Could not find instance loader in {[m.__name__ for m in _data_modules]}"

generated_dir = REPO_DIR / "data" / "generated"


def load_benchmark_instance(npz_path):
    '''Load an Instance from an .npz Path regardless of loader signature.'''
    sig = inspect.signature(_raw_loader)
    params = list(sig.parameters.keys())
    if params and params[0] == "name":
        return _raw_loader(npz_path.stem, generated_dir=npz_path.parent)
    return _raw_loader(npz_path)


# Warm up Numba JIT kernels on a small dummy schedule so solver timings stay clean
warmup_files = sorted(generated_dir.glob("tiny-*.npz"))
if warmup_files:
    _inst_w = load_benchmark_instance(warmup_files[0])
    _dummy_x = np.zeros(_inst_w.n_exams, dtype=np.int32)
    _ = proximity_cost(_inst_w, _dummy_x)
    _ = violation_weight(_inst_w, _dummy_x)

print(
    f"Environment & Numba JIT ready in {time.perf_counter() - t_env:.2f}s "
    f"(loader: {_raw_loader.__module__}.{_raw_loader.__name__})."
)
print("Registered metaheuristics:", ["ga", "ga_memetic", "pso", "pso_memetic", "aco", "aco_memetic"])

## 4. Benchmark Instance Validation & Structural Summary
The evaluation benchmark suite comprises 12 problem instances evenly distributed across four scale categories (`Tiny`, `Small`, `Medium`, and `Large`, with three instances per group), while tuning instances (`tune-small`, `tune-medium`, `tune-large`) remain strictly isolated to prevent data leakage.

The following cell validates the `CSR` structures and planted solutions (`check_planted=True`) for each evaluation instance and reports its dimensions, conflict density, and mean vertex degree.

In [ ]:
# Step 4: Load, validate, and summarize all 12 benchmark instances (excluding tune-*)
GROUP_ORDER = ["tiny", "small", "medium", "large"]
generated_dir = REPO_DIR / "data" / "generated"

instances_by_group = {grp: [] for grp in GROUP_ORDER}
for grp in GROUP_ORDER:
    for path in sorted(generated_dir.glob(f"{grp}-*.npz")):
        inst = load_benchmark_instance(path)
        if _raw_validator is not None:
            val_sig = inspect.signature(_raw_validator)
            if "check_planted" in val_sig.parameters:
                _raw_validator(inst, check_planted=True)
            else:
                _raw_validator(inst)
        if compute_instance_stats is not None:
            st = compute_instance_stats(inst)
            st_dict = st if isinstance(st, dict) else vars(st)
        else:
            st_dict = {}
        instances_by_group[grp].append((path.stem, inst, st_dict))

total_loaded = sum(len(v) for v in instances_by_group.values())
assert total_loaded == 12, f"Expected 12 evaluation instances, found {total_loaded}."

print(f"{'Group':<8} | {'Instance':<10} | {'Exams (N)':<9} | {'Slots (T)':<9} | {'Students (S)':<12} | {'Edges |E|':<10} | {'Density':<8} | {'Avg Deg':<8} | {'Status'}")
print("-" * 102)
for grp in GROUP_ORDER:
    for name, inst, st_dict in instances_by_group[grp]:
        n_edges = int(st_dict.get("n_edges", inst.n_edges))
        max_possible = inst.n_exams * (inst.n_exams - 1) / 2.0
        density = float(st_dict.get("density", n_edges / max(1.0, max_possible)))
        avg_deg = float(st_dict.get("avg_degree", (2.0 * n_edges) / max(1, inst.n_exams)))
        print(
            f"{grp.upper():<8} | {name:<10} | {inst.n_exams:<9} | {inst.n_slots:<9} | "
            f"{inst.n_students:<12} | {n_edges:<10} | {density:<8.3f} | {avg_deg:<8.2f} | VALID"
        )

## 5. Metaheuristic Execution Under Unified `FEE` Budget
This stage executes the base and memetic variants of all three metaheuristics (`ga`, `ga_memetic`, `pso`, `pso_memetic`, `aco`, `aco_memetic`) across the 12 benchmark instances in **`quick`** mode (two random seeds `0` and `1` per instance, with budgets of `150 FEE` for `Tiny`, `80` for `Small`, `60` for `Medium`, and `24` for `Large`; `144` total runs).

During execution, the runner automatically verifies:
1. Hard-constraint feasibility ($V(x) = 0$, i.e., `feasible = True`) for every produced timetable.
2. Strict compliance with the allocated `FEE` budget ceiling across all runs.
3. Persistence of raw execution records to `results/raw/walkthrough_runs.csv` alongside convergence trajectories (`history`) for visualization.

In [ ]:
# Step 5: Execute the 6 target algorithms across 12 instances (144 runs in quick mode)
import csv

MODE = "quick"  # Fast interactive mode (~30-45s total runtime)

MODE_CONFIG = {
    "quick": {
        "seeds": [0, 1],
        "budgets": {"tiny": 150.0, "small": 80.0, "medium": 60.0, "large": 24.0},
    },
    "pilot": {
        "seeds": [0, 1, 2, 3, 4],
        "budgets": {"tiny": 300.0, "small": 300.0, "medium": 300.0, "large": 80.0},
    },
}

ALGORITHMS = ["ga", "ga_memetic", "pso", "pso_memetic", "aco", "aco_memetic"]
seeds = MODE_CONFIG[MODE]["seeds"]
group_budgets = MODE_CONFIG[MODE]["budgets"]

raw_records = []
sample_histories = {}  # Store seed=0 convergence curves for representative instances
max_budget_overrun = 0.0

t_bench_start = time.perf_counter()

for grp in GROUP_ORDER:
    budget_fee = group_budgets[grp]
    grp_t0 = time.perf_counter()
    for inst_name, inst, _ in instances_by_group[grp]:
        for algo in ALGORITHMS:
            solver = get_solver(algo)
            for seed in seeds:
                res = solver.solve(inst, Budget(max_evals=budget_fee), seed=seed)
                overrun = float(res.evals) - float(budget_fee)
                max_budget_overrun = max(max_budget_overrun, overrun)
                # Budget.exhausted() checks evals >= max_evals before a step, so the final atomic
                # evaluation or delta step can overshoot by at most <= 1.0 FEE.
                assert overrun <= 1.0 + 1e-9, (
                    f"Budget overrun in {algo} on {inst_name} (seed={seed}): {res.evals} > {budget_fee}"
                )

                raw_records.append({
                    "group": grp,
                    "instance": inst_name,
                    "algorithm": algo,
                    "seed": seed,
                    "budget_fee": budget_fee,
                    "feasible": bool(res.feasible),
                    "best_f": float(res.best_f),
                    "evals": float(res.evals),
                    "elapsed": float(res.elapsed),
                })

                if seed == 0 and inst_name in {"tiny-a", "medium-a"}:
                    sample_histories[(inst_name, algo)] = res.history

    print(f"Completed group '{grp.upper():<6}' (budget={budget_fee:>5.1f} FEE) in {time.perf_counter() - grp_t0:.2f}s")

total_bench_time = time.perf_counter() - t_bench_start
feasible_count = sum(1 for r in raw_records if r["feasible"])

# Save raw CSV log for reproducibility
raw_csv_path = REPO_DIR / "results" / "raw" / "walkthrough_runs.csv"
raw_csv_path.parent.mkdir(parents=True, exist_ok=True)
with open(raw_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(raw_records[0].keys()))
    writer.writeheader()
    writer.writerows(raw_records)

print("-" * 78)
print(f"Total runs executed   : {len(raw_records)}")
print(f"Feasible solutions    : {feasible_count}/{len(raw_records)} ({100.0 * feasible_count / len(raw_records):.1f}%)")
print(f"Max (evals - budget)  : {max(0.0, max_budget_overrun):.6f} FEE (Within <= 1.0 FEE atomic step tolerance)")
print(f"Total wall-clock time : {total_bench_time:.2f}s")
print(f"Raw CSV saved to      : {raw_csv_path}")

## 6. Quantitative & Non-Parametric Statistical Analysis
To provide a scale-invariant comparison across heterogeneous problem sizes, results are aggregated at three levels:
1. **Overall Summary & Within-Instance Rank:** Solvers are ranked within each instance based on their median feasible proximity cost $f(x)$, and mean ranks are computed across all 12 instances (lower rank indicates better performance).
2. **Scale-Group Breakdown (`TINY`, `SMALL`, `MEDIUM`, `LARGE`):** Mean, median, interquartile range (`IQR`), wall-clock runtime, and consumed `FEE` within each homogeneous instance group.
3. **Non-Parametric Hypothesis Testing (`Friedman` & `Wilcoxon-Holm`):** The 12 instances are treated as independent comparison blocks to avoid pseudo-replication across seeds, with `Holm-Bonferroni` step-down $p$-value adjustment for pairwise comparisons against the top-ranked solver.

In [ ]:
# Step 6: Compute descriptive metrics, within-instance ranks, and Friedman / Wilcoxon-Holm tests
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

all_instances = [name for grp in GROUP_ORDER for name, _, _ in instances_by_group[grp]]

# Build instance x algorithm matrix of within-instance median feasible f(x)
median_matrix = np.zeros((len(all_instances), len(ALGORITHMS)), dtype=np.float64)
for i, inst_name in enumerate(all_instances):
    for j, algo in enumerate(ALGORITHMS):
        vals = [
            r["best_f"] for r in raw_records
            if r["instance"] == inst_name and r["algorithm"] == algo and r["feasible"]
        ]
        median_matrix[i, j] = float(np.median(vals)) if vals else float("inf")

# Compute within-instance ranks (1 = best/lowest cost) across the 12 instances
instance_ranks = np.zeros_like(median_matrix)
for i in range(len(all_instances)):
    instance_ranks[i] = stats.rankdata(median_matrix[i], method="average")

mean_ranks = instance_ranks.mean(axis=0)

print("=== 1. OVERALL DESCRIPTIVE SUMMARY ACROSS ALL 12 INSTANCES ===")
print(f"{'Algorithm':<14} | {'Runs':<5} | {'Feasible':<9} | {'Mean f*':<10} | {'Median f*':<10} | {'Mean Time(s)':<12} | {'Mean Rank'}")
print("-" * 86)

algo_summary = []
for j, algo in enumerate(ALGORITHMS):
    recs = [r for r in raw_records if r["algorithm"] == algo]
    feas_vals = [r["best_f"] for r in recs if r["feasible"]]
    feas_rate = 100.0 * len(feas_vals) / len(recs)
    mean_f = float(np.mean(feas_vals)) if feas_vals else float("inf")
    med_f = float(np.median(feas_vals)) if feas_vals else float("inf")
    mean_t = float(np.mean([r["elapsed"] for r in recs]))
    m_rank = float(mean_ranks[j])
    algo_summary.append((m_rank, algo, len(recs), feas_rate, mean_f, med_f, mean_t))

for m_rank, algo, n_runs, feas_rate, mean_f, med_f, mean_t in sorted(algo_summary):
    print(f"{algo:<14} | {n_runs:<5} | {feas_rate:>7.1f}% | {mean_f:<10.4f} | {med_f:<10.4f} | {mean_t:<12.4f} | {m_rank:<9.4f}")

print("\n=== 2. PERFORMANCE BREAKDOWN BY INSTANCE SCALE GROUP ===")
for grp in GROUP_ORDER:
    print(f"\n--- Group: {grp.upper()} (Budget = {group_budgets[grp]:.0f} FEE) ---")
    print(f"{'Algorithm':<14} | {'Mean f':<10} | {'Median f':<10} | {'IQR f':<10} | {'Mean Time(s)':<12} | {'Mean FEE'}")
    print("-" * 76)
    for algo in ALGORITHMS:
        recs = [r for r in raw_records if r["group"] == grp and r["algorithm"] == algo and r["feasible"]]
        vals = [r["best_f"] for r in recs]
        if vals:
            mean_v = float(np.mean(vals))
            med_v = float(np.median(vals))
            iqr_val = float(np.percentile(vals, 75) - np.percentile(vals, 25))
            mean_t = float(np.mean([r["elapsed"] for r in recs]))
            mean_fee = float(np.mean([r["evals"] for r in recs]))
        else:
            mean_v = med_v = iqr_val = mean_t = mean_fee = float("nan")
        print(
            f"{algo:<14} | {mean_v:<10.4f} | {med_v:<10.4f} | "
            f"{iqr_val:<10.4f} | {mean_t:<12.4f} | {mean_fee:<8.3f}"
        )

print("\n=== 3. NON-PARAMETRIC STATISTICAL TESTS (12 INSTANCE BLOCKS) ===")
friedman_stat, friedman_p = stats.friedmanchisquare(*median_matrix.T)
print(f"Friedman Omnibus Test : chi2 = {friedman_stat:.4f}, p-value = {friedman_p:.6f}")

best_idx = int(np.argmin(mean_ranks))
best_algo = ALGORITHMS[best_idx]
print(f"Top-ranked solver     : '{best_algo}' (Mean within-instance rank = {mean_ranks[best_idx]:.4f})")

pairwise_raw = []
for j, other_algo in enumerate(ALGORITHMS):
    if j == best_idx:
        continue
    diffs = median_matrix[:, best_idx] - median_matrix[:, j]
    if np.allclose(diffs, 0.0):
        w_stat, p_raw = 0.0, 1.0
    else:
        w_stat, p_raw = stats.wilcoxon(median_matrix[:, best_idx], median_matrix[:, j], alternative="two-sided")
    pairwise_raw.append((p_raw, other_algo, float(w_stat)))

# Holm-Bonferroni step-down adjustment
pairwise_raw.sort(key=lambda x: x[0])
m_tests = len(pairwise_raw)
print(f"\nPairwise Wilcoxon Signed-Rank Tests vs '{best_algo}' (Holm-Bonferroni adjusted):")
prev_adj = 0.0
for step, (p_raw, other_algo, w_stat) in enumerate(pairwise_raw):
    p_holm = min(1.0, max(prev_adj, p_raw * (m_tests - step)))
    prev_adj = p_holm
    sig_label = "Significant (p < 0.05)" if p_holm < 0.05 else "Not significant (p >= 0.05)"
    print(f"  {best_algo:<12} vs {other_algo:<12} | W = {w_stat:5.1f} | p_raw = {p_raw:.4f} | p_Holm = {p_holm:.4f} | {sig_label}")

## 7. Result Visualizations & Convergence Profiles
The final cell generates publication-ready figures directly from the recorded execution logs:
1. **Mean Within-Instance Rank & Feasibility Rate:** Compares relative solver rankings and confirms `100%` hard-constraint feasibility across all runs.
2. **Scale-Group Cost Distributions (`Boxplots`):** Displays median (orange line), mean (green triangle), and dispersion across `TINY`, `SMALL`, `MEDIUM`, and `LARGE` instances.
3. **`FEE` Budget Convergence Trajectories:** Tracks the evolution of best feasible Carter proximity cost `best_f_so_far` as a function of consumed `FEE` budget.

In [ ]:
# Step 7: Generate rank, feasibility, group boxplots, and FEE convergence figures
fig_dir = REPO_DIR / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams["axes.linewidth"] = 0.85

# --- Figure 1 & 2: Mean Within-Instance Rank & Feasibility Rate ---
sorted_summary = sorted(algo_summary, key=lambda x: x[0])
rank_labels = [x[1] for x in sorted_summary]
rank_vals = [x[0] for x in sorted_summary]
feas_vals = [x[3] for x in sorted_summary]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3), dpi=160)

bars1 = axes[0].bar(rank_labels, rank_vals, color="#1B365D", width=0.55)
axes[0].set_title("Average Within-Instance Rank (Lower is Better)", fontsize=11.5, fontweight="bold")
axes[0].set_ylabel("Mean Rank Across 12 Instances", fontsize=10)
axes[0].set_ylim(0, max(rank_vals) + 0.8)
axes[0].tick_params(axis="x", rotation=22)
axes[0].grid(axis="y", linestyle=":", alpha=0.5)
for bar, val in zip(bars1, rank_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.08, f"{val:.2f}", ha="center", va="bottom", fontsize=9)

bars2 = axes[1].bar(rank_labels, feas_vals, color="#2B6CB0", width=0.55)
axes[1].set_title("Feasibility Rate by Algorithm (V(x) = 0)", fontsize=11.5, fontweight="bold")
axes[1].set_ylabel("Feasible Runs (%)", fontsize=10)
axes[1].set_ylim(0, 112)
axes[1].tick_params(axis="x", rotation=22)
axes[1].grid(axis="y", linestyle=":", alpha=0.5)
for bar, val in zip(bars2, feas_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val:.0f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(fig_dir / "walkthrough_rank_feasibility.png", dpi=200)
plt.show()

# --- Figure 3: 2x2 Boxplots for TINY, SMALL, MEDIUM, and LARGE Groups ---
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2), dpi=160)
axes_flat = axes.flatten()
label_key = "tick_labels" if "tick_labels" in inspect.signature(axes_flat[0].boxplot).parameters else "labels"

for idx, grp in enumerate(GROUP_ORDER):
    ax = axes_flat[idx]
    box_data = [
        [r["best_f"] for r in raw_records if r["group"] == grp and r["algorithm"] == algo and r["feasible"]]
        for algo in ALGORITHMS
    ]
    ax.boxplot(
        box_data,
        **{label_key: ALGORITHMS},
        showmeans=True,
        meanprops=dict(marker="^", markerfacecolor="#16A34A", markeredgecolor="#16A34A", markersize=7),
        medianprops=dict(color="#EA580C", linewidth=1.6),
    )
    ax.set_title(f"Carter Proximity Cost f(x) — {grp.upper()} Group (Budget={group_budgets[grp]:.0f} FEE)", fontsize=10.8, fontweight="bold")
    ax.set_ylabel("Proximity Cost f(x)", fontsize=9.5)
    ax.tick_params(axis="x", rotation=22, labelsize=9)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(fig_dir / "walkthrough_group_boxplots.png", dpi=200)
plt.show()

# --- Figure 4: Convergence Trajectories Under Unified FEE Budget ---
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3), dpi=160)
colors = ["#1B365D", "#2563EB", "#0D9488", "#16A34A", "#D97706", "#DC2626"]

for ax_idx, inst_key in enumerate(["tiny-a", "medium-a"]):
    ax = axes[ax_idx]
    for algo, col in zip(ALGORITHMS, colors):
        hist = sample_histories.get((inst_key, algo))
        if hist is not None and len(hist) > 0:
            # Columns of Tracker.to_array(): [evals_used, iteration, best_f_so_far, seconds, diversity]
            evals_col = hist[:, 0]
            best_f_col = hist[:, 2]
            finite_mask = np.isfinite(best_f_col)
            if np.any(finite_mask):
                ax.step(evals_col[finite_mask], best_f_col[finite_mask], where="post", label=algo, color=col, linewidth=1.7)
    ax.set_title(f"Convergence Profile on '{inst_key}' (Seed 0)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Consumed Budget (FEE)", fontsize=9.8)
    ax.set_ylabel("Best Feasible f(x)", fontsize=9.8)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(fontsize=8.5, ncol=2)

plt.tight_layout()
plt.savefig(fig_dir / "walkthrough_convergence.png", dpi=200)
plt.show()

print(f"All walkthrough figures saved to: {fig_dir}")